In [38]:
import pandas as pd
import numpy as np

In [39]:
first_chunk = True

for chunk in pd.read_csv("../../../data/raw/PPIs/human_annotated_PPIs.txt", sep="\t",chunksize=50000, usecols=["symbol1", "symbol2", "kidney"]):
    colomns_to_keep = chunk[['symbol1','symbol2','kidney']]
    
    filtered = colomns_to_keep[colomns_to_keep['kidney'] >= 1]
    
    if first_chunk:
        filtered.to_csv('../../../data/kidney/processing/Kidney_filter.csv', index=False, mode='w', header=True)
        first_chunk = False
    else:
        filtered.to_csv('../../../data/kidney/processing/Kidney_filter.csv', index=False, mode='a', header=False)

In [ ]:
Kidney_filter = pd.read_csv('../../../data/kidney/processing/Kidney_filter.csv')
Kidney_filter


In [ ]:
Kidney_filter.isnull().any()

In [ ]:
Kidney_filter[Kidney_filter['symbol1'] == Kidney_filter['symbol2']].shape


In [43]:
Kidney_filter = Kidney_filter[Kidney_filter['symbol1'] != Kidney_filter['symbol2']]

In [ ]:
Kidney_filter.shape

In [ ]:
Kidney_filter.duplicated().sum()

In [46]:
Kidney_filter = Kidney_filter.drop_duplicates()


In [ ]:
Kidney_filter.shape

In [ ]:
pair = Kidney_filter[['symbol1','symbol2']]
print (pair)

In [ ]:
from operator import index


pair_filtered = pair.loc[pd.DataFrame(np.sort(pair.values,axis=1),index=pair.index).drop_duplicates().index]
print (pair_filtered)

In [ ]:
Kidney_filter = Kidney_filter.loc[pair_filtered.index]
Kidney_filter.dtypes

In [51]:
expressions = pd.read_csv('../../../data/raw/depmap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',nrows=0)

In [ ]:
print (expressions)

In [ ]:
gene_cols = expressions.columns[6:]   
gene_cols
  

In [54]:
exprn_genes = expressions.columns[6:].str.split(" (",regex=False).str[0]

In [ ]:
(~Kidney_filter["symbol1"].isin(exprn_genes)).sum()



In [ ]:
Kidney_filter[~Kidney_filter["symbol1"].isin(exprn_genes)]["symbol1"].nunique()

In [ ]:
(~Kidney_filter["symbol2"].isin(exprn_genes)).sum()


In [ ]:
Kidney_filter[~Kidney_filter["symbol2"].isin(exprn_genes)]["symbol2"].nunique()

In [ ]:
kidney_expr_filtered = Kidney_filter[Kidney_filter["symbol1"].isin(exprn_genes) & Kidney_filter["symbol2"].isin(exprn_genes)]

kidney_expr_filtered.shape

In [ ]:
pd.concat([kidney_expr_filtered["symbol1"], kidney_expr_filtered["symbol2"]]).nunique()

In [ ]:
gene = pd.read_csv("../../../data/raw/depmap/CRISPRGeneEffect.csv",nrows=0)
gene

In [ ]:
gene_lab = gene.columns[1:].str.split(" (",regex=False).str[0]
gene_lab

In [ ]:
ppi_genes = pd.concat([kidney_expr_filtered["symbol1"], kidney_expr_filtered["symbol2"]]).unique()

sum(pd.Series(ppi_genes).isin(gene_lab))


In [64]:
kidney_expr_filtered = kidney_expr_filtered.rename(columns={'symbol1':'A','symbol2':'B','kidney':'combined_score'})


In [65]:
for i in range(0,len(kidney_expr_filtered),100000):
    chunk = kidney_expr_filtered.iloc[i:i+100000]
    if i == 0:
        chunk.to_csv('../../../data/kidney/processing/kidney_PPI_final.csv',index=False,mode='w')
    else:
        chunk.to_csv('../../../data/kidney/processing/kidney_PPI_final.csv',index=False,mode='a',header=False)